# 04 多轮对话与流式响应

上一课你完成了工单结构校验。这一课把“接收一个完整回答”扩展成“带着历史提问，逐段显示回答，完成后才保存”。

这条能力会用于主线聊天后端：用户追问“那第二个呢？”时模型要看到先前对话；用户已看到半句话时，后台必须知道回答是否真正结束。

本课只有三个核心组，建议分三次学习，每次约15～20分钟：
1. 最近完整轮次与请求快照；
2. 片段累积与完成状态；
3. 完整回答与历史提交。

先阅读配套HTML微课，再在这里运行和填写代码。微课入口：`../lessons/0001-history-snapshot.html`、`0002-stream-lifecycle.html`、`0003-commit-complete-turn.html`。示例使用旅行助手，课堂练习使用食谱助手，作业使用客服助手。第1部分保持不变；从第2部分开始加入可手动开启的真实DeepSeek流。HTML交互、课堂检测和作业测试仍离线；只有标明真实请求的单元格会读取密钥、产生费用。教师提供接线模块，不代写你的练习和作业。

## 0. 先看全局地图：四份数据不能混在一起

| 数据 | 放什么 | 谁维护 | 什么时候使用 |
|---|---|---|---|
| history：已保存历史 | 过去成功完成的user/assistant轮次 | 应用 | 下一次组装请求时 |
| messages：请求快照 | system规则、选出的历史、本次问题 | 应用 | 本次请求发出时 |
| buffer：临时累积文本 | 当前收到的正文片段 | 应用 | 显示过程 |
| result：本次结果 | 完成状态和累积文本 | 应用 | 决定是否提交历史 |

“轮次”在本课精确定义为一条user加一条assistant；history不含system和当前未完成的问题。实际应用也可记录失败、工具消息等，但那不在本课简化契约内。

```text
T1 读取旧history
 → T2 按完整轮次裁剪，加入规则与本次问题，生成messages
 → T3 请求发出（可手动调用DeepSeek，或用固定事件离线复现）
 → T4 收到delta，累积并显示buffer，可重复多次
 → T5 收到正常完成done / 错误error / 未完成就断开
 → T6 得到result
 → T7 只有完整且非空，才把user+assistant写入新history
 → 下一次T1
```

裁剪本次请求不等于删除完整存档；显示半句话不等于保存成功回答。回忆上一课：完整JSON到达后才解析和校验，这里同样不能把一个流片段当成完整业务数据。

### 运行本课前：安装与检查

Python 3.10或以上，从仓库根目录打开终端，PowerShell/macOS/Linux均使用：

```powershell
cd "03_大模型应用开发/04_多轮对话与流式响应"
python -m pip install -r requirements.txt
python -c "import openai, dotenv, httpx, pytest; print(openai.__version__, pytest.__version__)"
```

python选择解释器；-m pip用该解释器运行包管理器；install安装；-r读取依赖文件。pytest用于离线测试；openai是调用DeepSeek兼容接口的SDK；python-dotenv读取本地.env；httpx是SDK的网络依赖，教师接线用它的异常类型。collections.abc、pathlib、sys来自标准库。Notebook需要安装时，在本课目录运行`%pip install -r requirements.txt`；它使用当前内核，必要时重启内核。Notebook内核与终端环境须一致。验证环境为openai 2.54.0、python-dotenv 1.2.2、httpx 0.28.1、pytest 8.4.2。安装失败先检查当前内核；不要在Notebook里粘贴密钥。

## 1. T1～T2：模型为什么能接住追问

在手动管理Chat Completions历史的方式中，每次请求由客户端提交messages。模型看到的是这次请求带来的内容；你在Python变量里保存的历史不会自动传过去。

[OpenAI会话状态指南](https://developers.openai.com/api/docs/guides/conversation-state)给出手动携带历史的方式，也介绍服务端状态管理。服务端保存是另一种接口能力，需要按接口传入会话标识或前次响应引用；保存或引用状态也不代表上下文无限长。本课只实现手动消息历史，不混用不同接口。

消息中的role、content沿用第二课，是API约定的字段，不是Python关键字：
- role：str，由客户端选择system/user/assistant；assistant表示先前模型回答；
- content：str，system规则由开发者提供，user来自用户，assistant来自模型完成后的正文；
- messages：客户端请求中的消息列表，顺序由应用组织，远端不会自动读取本地history。

看旅行示例：第一次问“去哪里”，第二次问“第二个怎么去”。如果丢掉上一轮，第二个失去指向。下面展示两种请求的内容，不让假模型编造结论。

In [1]:
travel_history = [
    {"role": "user", "content": "推荐两个杭州景点"},
    {"role": "assistant", "content": "西湖、灵隐寺。"},
]
current_question = "第二个怎么去？"
without_history = [{"role": "user", "content": current_question}]
with_history = [
    {"role": "system", "content": "你是旅行助手"},
    *travel_history,
    {"role": "user", "content": current_question},
]
print("缺上下文：", without_history)
print("带上下文：", with_history)

缺上下文： [{'role': 'user', 'content': '第二个怎么去？'}]
带上下文： [{'role': 'system', 'content': '你是旅行助手'}, {'role': 'user', 'content': '推荐两个杭州景点'}, {'role': 'assistant', 'content': '西湖、灵隐寺。'}, {'role': 'user', 'content': '第二个怎么去？'}]


`*travel_history`把列表元素展开到新列表中，它保留了每条消息的顺序；这仍共享原消息字典，下节会复制字典。这里只演示“带什么内容”。

### 1.1 最近两轮，不是最近两条消息

上下文窗口是一次模型处理能容纳的token预算；token是文本处理单位，不等于字符数。输入历史、系统规则、预留输出及模型特有开销都会占用预算。[官方上下文说明](https://developers.openai.com/api/docs/guides/conversation-state)

入门实验按轮次限制长度：最近n轮取2*n条消息。这个策略简单、可测试，但不能保证token不超限：一轮长文也可能很大，删除旧轮次也可能丢掉重要信息。实际项目需结合token计数、摘要或检索；本课不实现这些后续策略。

`history[-2:]`：Python列表切片，取最后2条；`history[:]`：复制整个列表。特别注意`history[-0:]`等于`history[0:]`，会取全部，不是取空列表。n=0必须明确保留零轮。

【先预测，再运行】先填写下面两个预测：history有两轮时，[-2:]返回几条？[-0:]返回几条？

In [4]:
predict_last_two = 2 # TODO：整数
predict_minus_zero = 4  # TODO：整数

In [5]:
try:
    assert "predict_last_two" in globals() and "predict_minus_zero" in globals(), "请先运行预测答题单元"
    assert predict_last_two is not None and predict_minus_zero is not None, "请先填写两个预测整数"
    assert type(predict_last_two) is int and predict_last_two == 2, "[-2:]场景：应取最后两条消息"
    assert type(predict_minus_zero) is int and predict_minus_zero == 4, "两轮历史[-0:]场景：-0等于0，会取全部四条"
    print("✅ 两项预测正确")
except AssertionError as _check_error:
    print("❌ 预测：", str(_check_error))
except Exception:
    print("❌ 预测：请确认变量存在，且填写了整数")

✅ 两项预测正确


In [6]:
two_turns = travel_history + [
    {"role": "user", "content": "哪个适合步行？"},
    {"role": "assistant", "content": "西湖适合。"},
]
print("最近一轮：", two_turns[-2:])
print("-0陷阱：", two_turns[-0:])
print("错误地只取最后一条：", two_turns[-1:])  # 只有assistant，缺配对问题

最近一轮： [{'role': 'user', 'content': '哪个适合步行？'}, {'role': 'assistant', 'content': '西湖适合。'}]
-0陷阱： [{'role': 'user', 'content': '推荐两个杭州景点'}, {'role': 'assistant', 'content': '西湖、灵隐寺。'}, {'role': 'user', 'content': '哪个适合步行？'}, {'role': 'assistant', 'content': '西湖适合。'}]
错误地只取最后一条： [{'role': 'assistant', 'content': '西湖适合。'}]


### 1.2 请求快照：改请求不应改存档

快照表示这次请求独立使用的一份数据。只复制列表还不够，因为列表里装的是可变字典。

标准库无需额外导入的`dict.copy()`无参数，返回新字典，浅拷贝其键值；本课消息值都是不可变str，所以已足够隔离。若以后content含嵌套对象，需要另选深拷贝策略。

`[message.copy() for message in history]`是列表推导式：依次复制每条消息，得到新列表。先看错误与正确对照：

In [7]:
bad_snapshot = travel_history[:]
bad_snapshot[0]["content"] = "被共享字典改了"
print("原历史也变了：", travel_history[0]["content"])
# 重建示例历史，隔离本示例的故意修改
travel_history = [
    {"role": "user", "content": "推荐两个杭州景点"},
    {"role": "assistant", "content": "西湖、灵隐寺。"},
]
good_snapshot = [message.copy() for message in travel_history]
good_snapshot[0]["content"] = "只改本次请求"
print("原历史保留：", travel_history[0]["content"])

原历史也变了： 被共享字典改了
原历史保留： 推荐两个杭州景点


### 课堂练习1：选择食谱助手的历史（5分钟）

目标：按完整轮次裁剪，并生成独立副本。
实现`recipe_recent(history: list[dict[str,str]], max_turns: int) -> list[dict[str,str]]`。
history保证按user/assistant成对排列、值都是str、不含system；max_turns保证是非负整数。
返回最近最多max_turns轮，保持顺序；0返回[]；历史不足保留全部；不得修改输入，返回字典也不能共享。无需写通用类型校验。

输入例子：
`[{"role":"user","content":"如何煮面"},{"role":"assistant","content":"水开后下面"}]`，max_turns=1应返回同样内容的新列表；max_turns=0返回[]。
本题不组装system或当前问题，先练习历史选择。

In [21]:
a=[10,20,30]
range(len(a)-2,len(a))

range(1, 3)

In [30]:
def recipe_recent(history: list[dict[str, str]], max_turns: int) -> list[dict[str, str]]:
    # TODO：选择最近完整轮次，并复制消息字典
    record = []
    n = len(history)
    if 2*max_turns>=n:
        return history
    for i in range(n-2*max_turns,n):
        record.append(history[i].copy())
    return record

In [31]:
try:
    assert "recipe_recent" in globals(), "请先运行课堂练习1答题单元"
    _check_history = [
        {"role": "user", "content": "煮面？"}, {"role": "assistant", "content": "先烧水。"},
        {"role": "user", "content": "放盐吗？"}, {"role": "assistant", "content": "少量。"},
    ]
    _check_original = [x.copy() for x in _check_history]
    for _check_limit, _check_expected in ((0, []), (1, _check_original[-2:]), (9, _check_original)):
        _check_result = recipe_recent(_check_history, _check_limit)
        assert isinstance(_check_result, list) and _check_result == _check_expected, f"max_turns={_check_limit}：完整轮次选择或消息顺序不正确"
    _check_result = recipe_recent(_check_history, 1)
    assert _check_history == _check_original, "构造快照不应修改原历史"
    _check_result[0]["content"] = "只改快照"
    assert _check_history == _check_original, "快照中的消息字典仍与原历史共享"
    assert recipe_recent([], 2) == [], "空历史应返回空列表"
    print("✅ 课堂练习1通过")
except NotImplementedError:
    print("❌ 课堂练习1尚未完成")
except AssertionError as _check_error:
    print("❌ 课堂练习1：", str(_check_error))
except Exception:
    print("❌ 课堂练习1：检查返回值应是消息字典列表，max_turns参数表示轮次")

✅ 课堂练习1通过


<details><summary>完成后查看课堂练习1参考答案</summary>

```python
def recipe_recent(history, max_turns):
    selected = history[-2 * max_turns:] if max_turns else []
    return [message.copy() for message in selected]
```

先按完整轮次选择，再复制字典。0是边界值，不能直接使用-0切片。

</details>

## 2. T3～T6：先看真实片段，再写收集器

这一部分只解决两件事：**已经收到哪些文字？这次回答是否正常完成？** 暂时不保存历史。

你问“先去西湖，再去灵隐寺，请复述顺序”。非流式返回完整回答后再使用；流式让你逐项收到新增文字，例如“先去”→“西湖”→“，再去灵隐寺。”。delta是新增部分，不是累计全文，也不保证一个片段就是一个字、token或完整句子。

```text
本次messages → DeepSeek → SDK逐项给出chunk
→ 取出新增正文 → 按序接起来 → 检查结束原因 → 本次result
```

先完成2.1～2.2的观察，休息后再做2.3～2.6的练习。不要一次背全部名词。

### 2.1 真实观察：每次究竟收到什么

以下是真实请求，不是伪代码。默认开关False，防止“运行全部”意外计费。你已允许真实实验，需要观察时手动改成True；每次执行该格发送一次请求，重复执行会再次计费。只提交课程编写的旅行问题，不提交私密内容。

配置模块复用第一课.env里的DEEPSEEK_API_KEY；本课默认模型为`deepseek-flash`，不继承或改写第一课的旧模型名。也可把本课.env.example复制为.env单独配置。HTML里不放密钥，不从浏览器直接调用。

[DeepSeek官方名称](https://api-docs.deepseek.com/news/news260910/)；[思考模式开关](https://api-docs.deepseek.com/guides/thinking_mode/)。本课明确禁用思考，只接正文；不能把reasoning_content当正文。

**运行前认识用到的对象与参数：**

| 名称 | 谁提供、类型、作用 |
|---|---|
| client | 教师配置函数创建的OpenAI客户端对象；创建时不发请求 |
| make_deepseek_client() | 本课模块函数，返回(client, model)元组；读取本地配置，不打印密钥 |
| client.chat.completions.create(...) | SDK方法，在T3发出请求；stream=True时返回可逐项读取并可close的流对象 |
| model / messages | 客户端传入的关键字参数：模型名str / 消息list |
| stream=True | bool关键字参数，要求逐步返回 |
| max_tokens=100 | int关键字参数，限制本次生成上限；不是正文保证长度 |
| extra_body={"thinking":{"type":"disabled"}} | SDK的dict关键字参数，携带DeepSeek专有开关；不是Python关键字 |
| timeout=30.0 / max_retries=0 | 配置函数传给OpenAI构造器的关键字参数：单次请求超时设置 / 不自动重试；不是严格的整段流总时长上限 |
| chunk.choices | 服务端响应经SDK解析后的选择列表；本课只请求一个回答，用[0]取它；非正文事件可跳过 |
| choice.delta.content | 服务端给出的新增正文，str或None，不是累计全文 |
| choice.finish_reason | 服务端给出的结束原因，str或None；None表示尚无结束原因 |

这些响应对象用点号访问属性；后面的练习输入是字典，用方括号访问，不能混用。[DeepSeek字段参考](https://api-docs.deepseek.com/api/create-chat-completion)

下面print里的`!r`让空字符串和空格以带引号的形式显现。stream.close()关闭流、释放连接；finally无论成功或失败都执行；client.close()释放客户端。异常只显示类别，不打印可能带请求资料的异常原文。

In [32]:
from pathlib import Path
import sys
from collections.abc import Iterable
from openai import OpenAIError
from httpx import HTTPError  # 网络读取异常，教师示范捕获

# 支持从课程目录、阶段目录或仓库根目录启动Notebook。
_course_candidates = [
    Path.cwd(),
    Path.cwd() / "04_多轮对话与流式响应",
    Path.cwd() / "03_大模型应用开发" / "04_多轮对话与流式响应",
]
_lesson_dir = next((p for p in _course_candidates if (p / "deepseek_stream.py").is_file()), None)
if _lesson_dir is None:
    raise RuntimeError("请从本课目录、阶段目录或仓库根目录启动Notebook，再运行本格。")
if str(_lesson_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(_lesson_dir.resolve()))
from deepseek_stream import make_deepseek_client, request_text_stream, deepseek_events
print("✅ 接线模块已导入；此格不读取密钥、不发送请求。")

✅ 接线模块已导入；此格不读取密钥、不发送请求。


In [34]:
RUN_REAL_REQUEST = False  # 观察真实流时，手动改为True；运行一次请求一次
live_question = "我想先去西湖，再去灵隐寺。只用一句话复述这个顺序。"
live_pieces = []
live_finish_reason = None
live_client = None
live_stream = None

if not RUN_REAL_REQUEST:
    print("⏭️ 未发送真实请求。需要观察时，把本格开关改为True；也可直接读下方已验证记录。")
else:
    try:
        live_client, live_model = make_deepseek_client()
        print("本次模型：", live_model, "；已禁用思考和自动重试。")
        live_stream = live_client.chat.completions.create(
            model=live_model,
            messages=[
                {"role": "system", "content": "你是旅行助手，简洁回答。"},
                {"role": "user", "content": live_question},
            ],
            stream=True,
            max_tokens=100,
            extra_body={"thinking": {"type": "disabled"}},
        )
        for live_chunk in live_stream:
            if not live_chunk.choices:
                continue
            live_choice = live_chunk.choices[0]
            live_part = live_choice.delta.content
            print(f"收到：text={live_part!r}，finish_reason={live_choice.finish_reason!r}")
            if live_part is not None:
                live_pieces.append(live_part)
            if live_choice.finish_reason is not None:
                live_finish_reason = live_choice.finish_reason
        print("已收集正文：", "".join(live_pieces))
        print("结束原因：", live_finish_reason, "；只有stop确认正常完成。")
    except ValueError as live_error:
        print("⚠️ 配置或数据读取错误；检查本地.env的密钥配置，不粘贴密钥到Notebook。")
    except (OpenAIError, HTTPError) as live_error:
        print("⚠️ 请求失败：", type(live_error).__name__, "；检查联网、密钥、额度，不自动重试。")
        print("失败前已收文字：", "".join(live_pieces))
    finally:
        if live_stream is not None:
            live_stream.close()
        if live_client is not None:
            live_client.close()

本次模型： deepseek-flash ；已禁用思考和自动重试。
收到：text=''，finish_reason=None
收到：text='先去'，finish_reason=None
收到：text='西湖'，finish_reason=None
收到：text='，'，finish_reason=None
收到：text='再去'，finish_reason=None
收到：text='灵'，finish_reason=None
收到：text='隐'，finish_reason=None
收到：text='寺'，finish_reason=None
收到：text='。'，finish_reason=None
收到：text=''，finish_reason='stop'
已收集正文： 先去西湖，再去灵隐寺。
结束原因： stop ；只有stop确认正常完成。


**怎样读输出，而不是只点运行：**

2026-10-10实测，第一轮前几段为`''`、`'先去'`、`'西湖'`、`'，'`，其finish_reason为None；最终正文“先去西湖，再去灵隐寺。”，最终原因stop。你可能得到不同分段和措辞，不能用固定片段数判断对错。

在输出中找到：一条空片段、一条新增正文、最后一条带结束原因的事件。若一次测试里没有空片段，也不代表失败。记录见`真实流观察记录.md`。

### 2.2 先把文字接起来：显示和保存到变量不是同一件事

显示：每段print，让用户先看到；收集：把每段存到pieces，最终有一整段str。只print不保留，后面没有完整正文可以提交。不要每次赋值覆盖已有正文。

先用固定列表练习拼接，不再发请求：

In [35]:
pieces = ["先去", "西湖", "，再去灵隐寺。"]
print("顺序拼接：", "".join(pieces))

parts = ["Hello", "", " world"]
print("保留空格：", "".join(parts))
print("错误清理：", "".join(part.strip() for part in parts))

bad_text = ""
for part in pieces:
    bad_text = part  # 错误：覆盖，丢失前面的文字
print("错误覆盖后：", bad_text)

顺序拼接： 先去西湖，再去灵隐寺。
保留空格： Hello world
错误清理： Helloworld
错误覆盖后： ，再去灵隐寺。


`pieces.append(text)`是list方法，追加一个元素，返回None；不能写`pieces = pieces.append(text)`。
`"".join(pieces)`是str方法，参数是字符串序列，返回连接后的str，空分隔符不额外插入字符。
delta里的None表示没有正文，跳过，不能放进join的字符串列表。`""`不增加文字，`" "`是正文空格，不逐段strip。

### 2.3 再确认：它有没有正常完成

下面四种情况，已经收到的正文都保留。差异在状态，不在“句子像不像完整”。status/text是我们应用生成的本地dict字段，不是请求参数，也不是HTTP状态码。

| 实际结束情况 | 本课转换 | 收集函数返回status |
|---|---|---|
| finish_reason="stop" | 发出done事件 | complete |
| finish_reason="length" | 保留正文，不发done/error，结束供给 | incomplete |
| 明确失败结束或SDK网络异常 | 发出error事件 | failed |
| 未收到结束原因，流已耗尽 | 不擅自补done | incomplete |

实测max_tokens=8的长行程问题只收到“以下是一份杭州三日旅行行程，”，结束原因length。**循环结束不等于回答成功。**明确失败也不需要抹掉已收文字；那是可查看的片段，不是成功存档。

【先预测，再运行】下面两个场景分别填写状态或正文，检测不联网。

In [38]:
predict_length_status = "incomplete"  # TODO：正文到半句，finish_reason="length"，本课status是什么？
predict_error_text = "建议先去西湖。"  # TODO：已收“建议先去西湖。”，随后error，结果text应是什么？

In [39]:
try:
    assert "predict_length_status" in globals() and "predict_error_text" in globals(), "先填写并运行预测单元"
    assert predict_length_status != "" and predict_error_text != "", "两个预测都需要填写"
    assert predict_length_status == "incomplete", "length场景：长度截断，不是正常完成或明确失败"
    assert predict_error_text == "建议先去西湖。", "error场景：保留之前收到的正文，不清空、不改字"
    print("✅ 流式两项预测通过")
except AssertionError as _check_error:
    print("❌ 流式预测：", str(_check_error))
except Exception:
    print("❌ 流式预测：先定义两个变量，并用字符串填写")

✅ 流式两项预测通过


### 2.4 为什么练习输入和刚才真实对象不一样

真实分段和网络不稳定，不适合用来判定你每次写的代码对不对。我们先把SDK对象转换为固定格式，再测试收集逻辑。这个转换叫**适配**：改变外部格式，不替你决定如何收集、如何提交。

```text
SDK对象：choice.delta.content="先去"
             ↓ 教师deepseek_events翻译
练习字典：{"kind":"delta", "text":"先去"}
             ↓ 你的recipe_stream收集
本次结果：{"status":"complete/incomplete/failed", "text":"已收正文"}
```

kind/text由本课定义，不是SDK字段或Python关键字，也不需要发给API。输入保证符合下表，不要求你做通用类型防御。

| 事件字典 | 类型与含义 | 你要做的事 |
|---|---|---|
| {"kind":"delta", "text":...} | kind为str；text为str或None，新增正文 | 按顺序累积，None跳过，空格保留 |
| {"kind":"done"} | kind为str，无text；正常完成 | 标记complete，立即停止 |
| {"kind":"error"} | kind为str，无text；明确失败 | 标记failed，保留片段，立即停止 |
| 没有事件可读且无done/error | 不是第四种字典，而是遍历耗尽 | 标记incomplete，保留片段 |

用列表演示输入即可理解练习。生成器是另一种输入供给方式：含yield的函数调用后得到一个生成器对象，每次取一项，函数继续到下一处yield再暂停；for逐项取数据。同一个生成器耗尽后不会从头重读，但重新调用生成器函数会创建新的生成器。

Iterable来自标准库collections.abc，类型标注表示可遍历的对象；本课包括列表、迭代器和生成器。iter(list)返回迭代器，next(iterator)取下一项，耗尽会抛StopIteration；for会自行处理耗尽。[Python原始说明](https://docs.python.org/3/tutorial/classes.html#generators)

下面原有最小流不联网。先看它如何一项项提供事件，再观察打印和收集在同一次遍历里完成。

In [40]:
from collections.abc import Iterable

def fake_travel_stream():
    yield {"kind": "delta", "text": "建议"}
    yield {"kind": "delta", "text": "先去西湖。"}
    yield {"kind": "done"}

for event in fake_travel_stream():
    print("收到事件：", event)
# 用next观察取一次才得到一次，不是调用函数就拿到完整列表
demo_iterator = iter([{"kind": "delta", "text": "一段"}, {"kind": "done"}])
print(next(demo_iterator))
print(next(demo_iterator))

收到事件： {'kind': 'delta', 'text': '建议'}
收到事件： {'kind': 'delta', 'text': '先去西湖。'}
收到事件： {'kind': 'done'}
{'kind': 'delta', 'text': '一段'}
{'kind': 'done'}


`print(text, end="", flush=True)`来自内置print函数：text为位置参数；end=""取消换行；flush=True尽快刷新输出；返回None。显示不会自动替你保存全文，所以同一循环还要append片段。

本地固定事件立即可读；真实流的for可能等待下一个网络片段。先遍历真实流打印、再遍历同一条流收集，会发现第二遍已经没有数据；不要用这种“两遍读取”方式。

In [41]:
pieces = []
for event in fake_travel_stream():
    if event["kind"] == "delta":
        pieces.append(event["text"])
        print(event["text"], end="", flush=True)
    elif event["kind"] == "done":
        break  # break是Python关键字，退出当前循环
travel_text = "".join(pieces)
print("\n完整文本：", travel_text)
# 变化：空片段不提供正文；空格是正文的一部分，不应strip掉
parts = ["第一段", "", " 第二段"]
print("保留间隔：", "".join(parts))
# 错误：每次覆盖会丢前面的片段
bad_text = ""
for part in parts:
    bad_text = part
print("覆盖后只剩最后一段：", bad_text)

建议先去西湖。
完整文本： 建议先去西湖。
保留间隔： 第一段 第二段
覆盖后只剩最后一段：  第二段


In [42]:
one_stream = fake_travel_stream()
print("第一遍：", list(one_stream))
print("第二遍：", list(one_stream))  # []，这个生成器已耗尽

第一遍： [{'kind': 'delta', 'text': '建议'}, {'kind': 'delta', 'text': '先去西湖。'}, {'kind': 'done'}]
第二遍： []


收集结果需同时返回status和text。以下只查看固定事件中的终止信号：done / 没有信号 / error。注意，没有信号不是自动done；只有done但没有正文，则是complete和空字符串，第3节仍不会保存它。

first done/error是本轮结束点，之后的事件不再读；正常结束不是等到“看起来足够长”。

In [43]:
for label, events in (
    ("正常", [{"kind": "delta", "text": "去西湖"}, {"kind": "done"}]),
    ("断开", [{"kind": "delta", "text": "去西"}]),
    ("失败", [{"kind": "delta", "text": "去"}, {"kind": "error"}]),
):
    terminal = None
    for event in events:
        if event["kind"] in ("done", "error"):
            terminal = event["kind"]
            break
    print(label, "结束信号：", terminal)  # done / None / error

正常 结束信号： done
断开 结束信号： None
失败 结束信号： error


### 2.5 课堂练习2：收集食谱流（8分钟）

目标：自己实现“按序收正文＋识别第一个终止事件”。你写的是接收员，不是API调用器。

实现`recipe_stream(events: Iterable[dict[str,str | None]]) -> dict[str,str]`。
输入是第2.4节三个事件之一组成的list或只能遍历一次的生成器；delta的text可为None或空串。无需验证其他输入类型，不访问choices或finish_reason，不读取密钥、不调用API。

按顺序累积正文，忽略None，保留空格。遇到done返回complete，遇error返回failed，均立即停止读取；无终止事件返回incomplete。返回恰好status、text两个键，它们的值都是str。

输入：`[{"kind":"delta","text":"先"},{"kind":"delta","text":"烧水"},{"kind":"done"}]`
输出：`{"status":"complete","text":"先烧水"}`。
只收到“先”就耗尽，输出`{"status":"incomplete","text":"先"}`；已收“先”随后error，输出`{"status":"failed","text":"先"}`。

这道练习只收集结果，不保存历史。先运行你的答题单元，再运行下面离线检测；通过后再去第2.6节接真实流。参考答案折叠，先独立写。

In [47]:
def recipe_stream(events: Iterable[dict[str, str | None]]) -> dict[str, str]:
    # TODO：单次遍历，累积片段并识别终止状态
    pieces = []
    terminal = None
    for event in events:
        if event["kind"] == "delta" and event["text"] is not None:
            pieces.append(event["text"])
        elif event["kind"] in ["done","error"]:
            terminal = event["kind"]
            break 
    if terminal is None:
        return {"status":"incomplete","text":"".join(pieces)}
    elif terminal == "done":
        return {"status":"complete","text":"".join(pieces)}
    else:
        return {"status":"failed","text":"".join(pieces)}        

In [48]:
try:
    assert "recipe_stream" in globals(), "请先运行课堂练习2答题单元"
    for _check_events, _check_expected, _check_name in (
        ([{"kind": "delta", "text": "先"}, {"kind": "delta", "text": "烧水"}, {"kind": "done"}], {"status": "complete", "text": "先烧水"}, "正常正文和done"),
        ([{"kind": "delta", "text": None}, {"kind": "delta", "text": ""}, {"kind": "delta", "text": " 盐 "}, {"kind": "done"}], {"status": "complete", "text": " 盐 "}, "空片段与空格"),
        ([{"kind": "delta", "text": "先"}], {"status": "incomplete", "text": "先"}, "没有done的断开"),
        ([{"kind": "delta", "text": "先"}, {"kind": "error"}, {"kind": "done"}], {"status": "failed", "text": "先"}, "error优先终止"),
        ([{"kind": "done"}], {"status": "complete", "text": ""}, "空正文正常完成"),
    ):
        assert recipe_stream(iter(_check_events)) == _check_expected, f"{_check_name}：status或text不满足规则"
    _check_iterator = iter([{"kind": "done"}, {"kind": "delta", "text": "不要读"}])
    assert recipe_stream(_check_iterator) == {"status": "complete", "text": ""}, "done时应返回完成"
    assert next(_check_iterator)["text"] == "不要读", "done之后不应继续读取"
    print("✅ 课堂练习2通过")
except NotImplementedError:
    print("❌ 课堂练习2尚未完成")
except AssertionError as _check_error:
    print("❌ 课堂练习2：", str(_check_error))
except Exception:
    print("❌ 课堂练习2：检查None片段应忽略，返回值应是含status和text的字典；只遍历流一次")

✅ 课堂练习2通过


<details><summary>完成后查看课堂练习2参考答案</summary>

```python
def recipe_stream(events):
    pieces = []
    for event in events:
        if event["kind"] == "delta":
            if event["text"] is not None:
                pieces.append(event["text"])
        elif event["kind"] == "done":
            return {"status": "complete", "text": "".join(pieces)}
        elif event["kind"] == "error":
            return {"status": "failed", "text": "".join(pieces)}
    return {"status": "incomplete", "text": "".join(pieces)}
```

循环正常耗尽不是done；保留partial text可用于显示，但不应被当作完整回答。

</details>

### 2.6 把你写好的练习2接到真实DeepSeek

先通过练习2检测，再运行本节。你原来的recipe_stream无需增加API参数：教师接线模块替你把真实对象变成事件字典，收集仍由你的函数完成。

request_text_stream(client, model, messages, max_tokens=...)来自本课模块，等价于2.1节的SDK请求设置，返回真实流；deepseek_events(stream, diagnostics)返回事件生成器，diagnostics字典接收结束原因和异常类型，不放密钥或异常原文。适配器遇stop发done；length不发done；明确失败或SDK异常发error。finally关闭上游流。

本节独立开关RUN_REAL_COLLECT：一次执行新发一次请求，不是重读2.1节已经耗尽的流。建议先保持REAL_MAX_TOKENS=100，问题要求一句话。观察截断时可改为8并改问长行程；是否length以服务端实际返回为准。

events.close()关闭这个生成器，触发教师模块finally及时关闭上游；外层finally同时关闭流和客户端，提前返回或错误也不泄漏连接。这是教师接线代码，不要求你重写网络异常处理或自动重连。

In [49]:
RUN_REAL_COLLECT = False # 完成练习2后手动开启；一次执行一个新请求
REAL_MAX_TOKENS = 100
travel_live_question = "我想先去西湖，再去灵隐寺。只用一句话复述这个顺序。"
travel_live_result = None
travel_live_diagnostics = {}
_live_client = None
_live_stream = None
_live_events = None

if not RUN_REAL_COLLECT:
    print("⏭️ 真实收集关闭；先完成练习2并运行检测。")
else:
    try:
        assert "recipe_stream" in globals(), "请先运行练习2答题单元"
        _probe = recipe_stream(iter([{"kind": "delta", "text": "检查"}, {"kind": "done"}]))
        assert _probe == {"status": "complete", "text": "检查"}, "先通过练习2离线检测，再接真实流"
        _live_client, _live_model = make_deepseek_client()
        _live_stream = request_text_stream(_live_client, _live_model, [
            {"role": "system", "content": "你是旅行助手，简洁回答。"},
            {"role": "user", "content": travel_live_question},
        ], max_tokens=REAL_MAX_TOKENS)
        _live_events = deepseek_events(_live_stream, travel_live_diagnostics)
        travel_live_result = recipe_stream(_live_events)
        print("原始结束原因/异常类别：", travel_live_diagnostics)
        print("你的收集结果：", travel_live_result)
    except (AssertionError, NotImplementedError) as _live_error:
        print("⚠️ 先完成练习2并通过检测；本次不再继续请求。")
    except ValueError as _live_error:
        print("⚠️ 配置或数据错误；检查本地.env与练习2，不输出异常原文。")
    except (OpenAIError, HTTPError) as _live_error:
        print("⚠️ 请求创建失败：", type(_live_error).__name__, "；不自动重试。")
    finally:
        if _live_events is not None:
            _live_events.close()
        if _live_stream is not None:
            _live_stream.close()
        if _live_client is not None:
            _live_client.close()

原始结束原因/异常类别： {'finish_reason': 'stop', 'error_type': None}
你的收集结果： {'status': 'complete', 'text': '您先游览西湖，然后前往灵隐寺。'}


## 3. T7：将真实结果保存成一轮，再追问

本节只解决“什么时候把本次问题与回答加入history”。第2节已经收到了result，不能在接收每个delta时就往成功历史写一条assistant。

### 3.1 先明确存档规则

本课存档策略：result["status"]为complete且text包含非空白字符时，提交一条user和一条assistant；否则旧成功历史保持原内容。真实失败片段可以另记日志，这里不是要求清除所有失败记录。

```text
result → 判断complete且正文非空白 → 复制旧history → 追加user+assistant
                                                  ↓
                               下一次messages携带这一轮，理解追问
```

注意result是dict，用result["status"]访问，不写result.status。
下面仍用固定旅行结果演示，不发请求；完成练习3后，第3.3节把同一策略接到你刚收集的真实结果。

str.strip()无参数返回去掉两端空白的新str，只用来判断是否为空，保存时保留原文。
不要构造请求时就写入history，再在结束时写入一次；本次问题在请求里出现一次，在完成的存档轮次里出现一次，属于两个不同对象。

In [50]:
travel_result = {"status": "complete", "text": " 坐公交前往。 "}
old_saved = [message.copy() for message in travel_history]
new_saved = [message.copy() for message in old_saved]
if travel_result["status"] == "complete" and travel_result["text"].strip():
    new_saved.append({"role": "user", "content": current_question})
    new_saved.append({"role": "assistant", "content": travel_result["text"]})
print("旧存档长度：", len(old_saved))  # 2
print("新存档长度：", len(new_saved))  # 4
# 变化和反例：有字不等于完成，完成但没字也不能保存
for result in (
    {"status": "incomplete", "text": "坐公交"},
    {"status": "failed", "text": "坐"},
    {"status": "complete", "text": "   "},
):
    print(result, "是否允许保存：", result["status"] == "complete" and bool(result["text"].strip()))
# bool(value)是内置转换，空字符串为False，非空为True；len(list)返回元素数量整数

旧存档长度： 2
新存档长度： 4
{'status': 'incomplete', 'text': '坐公交'} 是否允许保存： False
{'status': 'failed', 'text': '坐'} 是否允许保存： False
{'status': 'complete', 'text': '   '} 是否允许保存： False


### 3.2 课堂练习3：提交食谱的一轮对话（5分钟）

实现`recipe_commit(history, question, result) -> list[dict[str,str]]`。
history是完整轮次列表，question是str，result是练习2返回的status/text字典。
complete且text不全为空白时追加user(question)、assistant(text)，其余返回独立复制的旧历史。
任何分支都不得修改输入；不得先把question写进输入history，也不得把text清理后保存。

例：history=[]，question="如何煮面"，result={"status":"complete","text":" 先烧水。 "}
应返回`[{"role":"user","content":"如何煮面"},{"role":"assistant","content":" 先烧水。 "}]`。
result改为incomplete时，应返回[]。不使用数据库。

In [61]:
def recipe_commit(
    history: list[dict[str, str]], question: str, result: dict[str, str]
) -> list[dict[str, str]]:
    # TODO：只有完整且非空时，追加一对消息到历史副本
    new_history=[message.copy() for message in history]
    if result["status"] == "complete" and  result["text"].strip():
        a = {"role" : "user", "content" : question}
        b = {"role" : "assistant", "content" : result["text"]}
        new_history.append(a)
        new_history.append(b)
    return new_history



In [62]:
try:
    assert "recipe_commit" in globals(), "请先运行课堂练习3答题单元"
    _check_history = [{"role": "user", "content": "放盐吗"}, {"role": "assistant", "content": "少量"}]
    _check_original = [x.copy() for x in _check_history]
    _check_result = recipe_commit(_check_history, "煮面呢", {"status": "complete", "text": " 先烧水 "})
    assert _check_result == _check_original + [{"role": "user", "content": "煮面呢"}, {"role": "assistant", "content": " 先烧水 "}], "成功场景：应追加一对消息，保留原文空格"
    assert _check_history == _check_original, "提交不能原地改输入history"
    _check_result[0]["content"] = "改副本"
    assert _check_history == _check_original, "旧消息字典应复制"
    for _check_status, _check_text in (("incomplete", "半句"), ("failed", "片段"), ("complete", "  "), ("complete", "")):
        _check_result = recipe_commit(_check_history, "问题", {"status": _check_status, "text": _check_text})
        assert _check_result == _check_original, f"{_check_status}/{_check_text!r}场景：不能新增成功轮次"
        _check_result[0]["content"] = "改返回值"
        assert _check_history == _check_original, "未提交分支也需要复制历史字典"
    print("✅ 课堂练习3通过")
except NotImplementedError:
    print("❌ 课堂练习3尚未完成")
except AssertionError as _check_error:
    print("❌ 课堂练习3：", str(_check_error))
except Exception:
    print("❌ 课堂练习3：检查返回列表、完整且非空的提交条件及消息字典复制")

✅ 课堂练习3通过


<details><summary>完成后查看课堂练习3参考答案</summary>

```python
def recipe_commit(history, question, result):
    saved = [message.copy() for message in history]
    if result["status"] == "complete" and result["text"].strip():
        saved.append({"role": "user", "content": question})
        saved.append({"role": "assistant", "content": result["text"]})
    return saved
```

保存的是完整正文，不是每个片段；strip只用于检查，不用于改正文。

</details>

### 3.3 用你保存的真实一轮，问“第二站是哪里”

前提：第2.6节已经产生travel_live_result，练习2和3离线检测通过。此格先调用你写的recipe_commit：成功且非空才生成包含两条消息的新历史。

如果第2.6节结果为failed/incomplete或空正文，此格不发追问，也不伪造历史。成功时打印要发的messages，再发送一次简短追问；教师只提供调用连接，不替你完成函数。

RUN_REAL_FOLLOWUP默认False，开启后一次执行最多一个新请求。新messages：system→上轮user→上轮assistant→本次user。不需要模型猜“第二站”指什么，它可以读取上一轮正文。回答措辞不固定，不断言字面完全一致。

只使用课程例子。如果把问题改成私人资料，这些文字会发送到DeepSeek。此格只在内存演示，不写数据库，不把密钥或完整SDK输出保存到Notebook。

In [63]:
RUN_REAL_FOLLOWUP = False  # 完成练习2和3后手动开启；最多新发一次请求
followup_question = "第二站是哪里？只回答地点名称。"
followup_result = None
_follow_client = None
_follow_stream = None
_follow_events = None

if not RUN_REAL_FOLLOWUP:
    print("⏭️ 真实追问关闭；先完成练习3及检测。")
else:
    try:
        assert "recipe_commit" in globals() and "recipe_stream" in globals(), "先运行练习2和3答题单元"
        assert isinstance(globals().get("travel_live_result"), dict), "先运行第2.6节真实收集"
        saved_live_history = recipe_commit([], travel_live_question, travel_live_result)
        _can_save = travel_live_result["status"] == "complete" and bool(travel_live_result["text"].strip())
        if not _can_save:
            assert saved_live_history == [], "未完成或空回答不能写入成功历史"
            print("本次结果未满足成功提交条件，不保存、不发送追问。")
        else:
            assert saved_live_history == [
                {"role": "user", "content": travel_live_question},
                {"role": "assistant", "content": travel_live_result["text"]},
            ], "先通过练习3检测，确保准确提交一对消息"
            followup_messages = [
                {"role": "system", "content": "你是旅行助手，简洁回答。"},
                *[message.copy() for message in saved_live_history],
                {"role": "user", "content": followup_question},
            ]
            print("本次发送内容：", followup_messages)
            _follow_client, _follow_model = make_deepseek_client()
            _follow_stream = request_text_stream(_follow_client, _follow_model, followup_messages, max_tokens=60)
            _follow_events = deepseek_events(_follow_stream)
            followup_result = recipe_stream(_follow_events)
            print("追问结果：", followup_result)
            after_followup = recipe_commit(saved_live_history, followup_question, followup_result)
            print("追问前消息数：", len(saved_live_history), "；追问后消息数：", len(after_followup))
    except (AssertionError, NotImplementedError) as _follow_error:
        print("⚠️ 先通过练习2、3检测，并运行第2.6节得到真实结果；不继续请求。")
    except ValueError as _follow_error:
        print("⚠️ 配置或数据错误；检查本地.env与练习2、3，不输出异常原文。")
    except (OpenAIError, HTTPError) as _follow_error:
        print("⚠️ 请求创建失败：", type(_follow_error).__name__, "；不自动重试。")
    finally:
        if _follow_events is not None:
            _follow_events.close()
        if _follow_stream is not None:
            _follow_stream.close()
        if _follow_client is not None:
            _follow_client.close()

本次发送内容： [{'role': 'system', 'content': '你是旅行助手，简洁回答。'}, {'role': 'user', 'content': '我想先去西湖，再去灵隐寺。只用一句话复述这个顺序。'}, {'role': 'assistant', 'content': '您先游览西湖，然后前往灵隐寺。'}, {'role': 'user', 'content': '第二站是哪里？只回答地点名称。'}]
追问结果： {'status': 'complete', 'text': '灵隐寺。'}
追问前消息数： 2 ；追问后消息数： 4


## 4. 课末作业：客服助手的多轮处理

完成`conversation_flow.py`中的三个TODO，只改函数体，不改接口或测试。真实API接线属于教师示范，不增加作业负担；你的三个函数仍是可离线测试的核心。输入都满足下面的类型约定；不需要字符串清理、密钥、SDK、网络、数据库或类型防御判断。

### 任务1：build_request_messages(rule, history, question, max_turns)

| 参数 | 类型 | 内容/提供者 |
|---|---|---|
| rule | str | 开发者提供的system规则 |
| history | list[dict[str,str]] | 应用保存的完整user/assistant轮次，不含system或本次问题 |
| question | str | 用户本次问题，不在history中提前保存 |
| max_turns | int，非负 | 应用设定的最多历史轮次，必须显式传入 |

返回list[dict[str,str]]：一条system → 最近最多max_turns轮历史 → 一条本次user。max_turns=0只保留system和本次user；不足则保留全部旧轮次。请求只新增本次问题一次；同样文字的旧问题不可去重删除。不修改history；返回的旧消息字典必须复制。

输入：rule="客服规则"，history=[{"role":"user","content":"支持导出吗"},{"role":"assistant","content":"支持PDF"}]，question="怎么操作"，max_turns=1。
输出：
```python
[
    {"role": "system", "content": "客服规则"},
    {"role": "user", "content": "支持导出吗"},
    {"role": "assistant", "content": "支持PDF"},
    {"role": "user", "content": "怎么操作"},
]
```

### 任务2：collect_stream(events)

events类型Iterable[dict[str,str | None]]，可能是只能消费一次的迭代器；只含第2节的规范化delta/done/error事件。
返回dict[str,str]，恰好status和text：状态为complete/incomplete/failed。正文按收到顺序拼接，None忽略，空格保留。第一个done或error立即结束读取；没收到终止事件就是incomplete。

输入`iter([{"kind":"delta","text":"点击"},{"kind":"delta","text":"导出"},{"kind":"done"}])`。
输出`{"status":"complete","text":"点击导出"}`。
只有done的流返回complete和空字符串；空流返回incomplete和空字符串；error后忽略后续done。不假定每段对应一个字、一个token或完整JSON。

### 任务3：commit_turn(history, question, result)

history和question同任务1；result是任务2返回的字典。返回新的list[dict[str,str]]。
只在complete且text包含非空白字符时，向旧历史副本追加一条user(question)、一条assistant(text)；其余仅返回旧历史副本。正文保留原空格。不修改任何输入，不重新裁剪存档，不再次调用模型。

例：history=[]，question="如何导出"，result={"status":"complete","text":"点击导出"}。
输出：
```python
[
    {"role": "user", "content": "如何导出"},
    {"role": "assistant", "content": "点击导出"},
]
```
result改为{"status":"incomplete","text":"点击"}，输出[]。

这个函数每次调用代表一个独立待提交轮次；本课不要求识别重复请求ID。输入question即使重复同样文字，也可能是合法新问题，不能按文字去重。实际应用的重试防重复还需要请求标识与事务控制。

### 契约与核心目标对应

| 核心目标 | 作业 | 测试 |
|---|---|---|
| 顺序、完整轮次、上下文选择、快照 | 任务1 | 6项request测试 |
| 单次读取、片段拼接、终止状态 | 任务2 | 7项stream测试 |
| 成功提交、失败不污染、跨轮组合 | 任务3 | 4项commit及1项flow测试 |

共18项离线测试，核心知识18项，通用类型边界专项0项。初始骨架全部因TODO未完成而失败是预期。
在本课目录运行：
```powershell
python -m pytest -q test_conversation_flow.py
```
分组运行可用`-k test_request_`（6项）、`-k test_stream_`（7项）、`-k test_commit_`（4项）、`-k test_flow_`（1项）。先完成任务1和2，再完成任务3。

## 5. 理解题：先凭记忆作答

每题选一个字母，填入紧下方answers。检测只解释答错的题。题目对应第1、1.1、2.3、3.1、0/2节。

**第1题** 手动管理历史时，要让模型理解上一轮的追问，应怎么做？

A. 只发送当前问题即可保持上下文

B. 随请求发送旧的轮次和当前问题

C. 只保存本地历史即可保持上下文

D. 把所有的旧回答都当作用户提问

**第2题** 已保存两轮对话，本次选择零轮历史，正确行为是什么？

A. 保留最后两条历史消息

B. 保留最后一条历史消息

C. 保留规则和本次的问题

D. 保留所有的旧历史消息

**第3题** 已经收到正文片段，事件耗尽却没有done或error，结果是什么？

A. 保留片段并标记未完成

B. 保留片段并标记已完成

C. 清空片段并标记已完成

D. 补上结束并标记已完成

**第4题** 什么时候允许向本课的成功历史新增一轮？

A. 收到任意正文就保存这一轮

B. 收到失败事件就保存这一轮

C. 正文长度足够就保存这一轮

D. 正常完成非空才保存这一轮

**第5题** 与上一课结合，流式传回的JSON片段应怎么处理？

A. 每次收到新片段就作完整校验

B. 正常结束拼接后再作完整校验

C. 流自然耗尽之后就作完整校验

D. 片段成功显示之后作完整校验

In [65]:
answers = {
    "q1": "B", "q2": "C", "q3": "A", "q4": "D", "q5": "B",
}

In [66]:
try:
    assert "answers" in globals() and isinstance(answers, dict), "请先运行理解题答题单元，answers应为字典"
    _check_key = {"q1":"B","q2":"C","q3":"A","q4":"D","q5":"B"}
    _check_explain = {"q1":"手动管理时，模型通过本次messages读取选中的历史，保存本地变量不等于发送。","q2":"max_turns=0不发送旧轮次，但仍发送system和本次user；[-0:]会错误保留全部历史。","q3":"流耗尽不等于正常完成，没有done或error时应为incomplete并保留临时文本。","q4":"提交要求complete且正文非空白，并一起追加user与assistant，失败或半句不能提交。","q5":"单段通常不是完整JSON，应先确认正常完成并拼接，再进行JSON解析与字段校验。"}
    _check_complete = True
    for _check_q, _check_correct in _check_key.items():
        _check_answer = answers.get(_check_q, "")
        if not isinstance(_check_answer, str) or _check_answer.strip().upper() not in ("A", "B", "C", "D"):
            print("❌", _check_q, "尚未作答或不是单个A/B/C/D字母")
            _check_complete = False
        elif _check_answer.strip().upper() != _check_correct:
            print("❌", _check_q, _check_explain[_check_q])
            _check_complete = False
    if _check_complete:
        print("✅ 理解题5题全部正确")
except AssertionError as _check_error:
    print("❌ 理解题：", str(_check_error))
except Exception:
    print("❌ 理解题：请检查answers字典和每题单个字母")

✅ 理解题5题全部正确


## 6. 第二天再做：隔天回忆

今天做完后停止。**第二天**先关掉参考答案，复制一份你的练习到临时草稿，把三个函数的实现清空，再凭记忆重写并运行检测；保留原讲义中的完成答案，不覆盖它。
在草稿中将answers清空，再答5题。先画T1～T7，再检查“在哪一步加当前问题、在哪一步拼正文、在哪一步提交历史”。

交替回忆上一课：如果流式传来`'{"summary":'`，现在能否做完整工单校验？补全、正常结束后又要经过哪两步？用上一课的速查核对JSON解析→字段校验。只有确认完整正文仍需校验，本课complete只是传输状态，不证明业务内容正确。

可在3～7天后再做一次。当前课堂测试通过不等于长期记住，隔天结果再由你提供验收。

隔天复习先不开真实开关，凭记忆写出stop/length如何映射本课状态，再用固定事件验证；不要把付费请求次数当作复习成绩。

## 7. 速查与提问入口

本课`速查_多轮与流式.md`只保留接口、时间线和排错索引。
阶段`../reference/0001-conversation-flow.html`供浏览器快速查阅；
`../术语表.md`固定术语，后续课程沿用。
有疑问直接把章节、代码或检测提示发给我，我会按具体卡点继续讲。

参考来源（DeepSeek及SDK追加核对：2026-10-10；其余2026-10-09）：
- [DeepSeek V4.1-Flash模型名](https://api-docs.deepseek.com/news/news260910/)
- [DeepSeek流式字段](https://api-docs.deepseek.com/api/create-chat-completion)
- [DeepSeek思考模式](https://api-docs.deepseek.com/guides/thinking_mode/)
- [OpenAI会话状态](https://developers.openai.com/api/docs/guides/conversation-state)
- [OpenAI流式响应](https://developers.openai.com/api/docs/guides/streaming-responses)
- [Chat Completions事件字段](https://developers.openai.com/api/reference/resources/chat/subresources/completions/streaming-events)
- [Python生成器](https://docs.python.org/3/tutorial/classes.html#generators)